Viewed backprop.ipynb:1-203

本篇对 [backprop.ipynb](file:///Users/songchao/WorkSpace/PythonProject/DeepLearn/pytorch/chapter_multilayer-perceptrons/backprop.ipynb)（前向传播、反向传播与计算图）的核心内容进行系统总结：

---

### 一、 核心主题与背景

* **核心目标**：揭示深度学习框架中「自动求导（Auto-differentiation / Autograd）」底层的数学机理与计算流程。
* **基本设定**：以一个带有 **L2 正则化（权重衰减）的单隐藏层多层感知机（MLP）** 为例（为简化推导，暂不包含偏置项），分析单个样本在训练时的前向与反向推导。

---

### 二、 前向传播 (Forward Propagation)

**定义**：沿着神经网络结构，**从输入层到输出层**按顺序计算并存储各层中间变量及最终目标函数的过程。

#### 1. 前向计算流程
设输入样本 `x ∈ ℝᵈ`，隐藏层权重 `W_1 ∈ ℝ^(h×d)`，输出层权重 `W_2 ∈ ℝ^(q×h)`：

```text
1. 隐藏层未激活输出（中间变量）：
   z = W_1 · x                 (z ∈ ℝʰ)

2. 隐藏层激活值（中间变量）：
   h = φ(z)                    (h ∈ ℝʰ，φ 为激活函数)

3. 输出层预测值：
   o = W_2 · h                 (o ∈ ℝ^q)

4. 样本损失（Loss）：
   L = l(o, y)                 (y 为真实标签，L 为标量)

5. L2 正则化惩罚项：
   s = (λ / 2) · (||W_1||_F² + ||W_2||_F²)

6. 最终优化目标函数（Objective）：
   J = L + s
```

#### 2. 前向计算图 (Computational Graph)
* **节点类型**：圆圈表示**算子/操作符**（如矩阵乘法、激活函数、加法），方块表示**数据/变量**（输入、权重、中间激活值、损失）。
* **数据流向**：自底向上、从左向右流动。

---

### 三、 反向传播 (Backpropagation)

**定义**：根据微积分的**链式法则（Chain Rule）**，**从输出层到输入层**逆向遍历计算图，计算目标函数关于网络各参数及中间变量的梯度的过程。

#### 梯度推导链条（自顶向下）：

```text
1. 目标函数关于损失项与正则项的梯度：
   ∂J/∂L = 1， ∂J/∂s = 1

2. 关于输出层变量 o 的梯度：
   ∂J/∂o = ∂L/∂o ∈ ℝ^q

3. 正则项关于权重矩阵的梯度：
   ∂s/∂W_1 = λ · W_1
   ∂s/∂W_2 = λ · W_2

4. 输出层权重 W_2 的完整梯度：
   ∂J/∂W_2 = (∂J/∂o) · hᵀ + λ · W_2      (结果维度：q × h)

5. 隐藏层激活变量 h 的梯度：
   ∂J/∂h = (W_2)ᵀ · (∂J/∂o)              (结果维度：h × 1)

6. 隐藏层未激活变量 z 的梯度（由于 φ 是逐元素作用，此处采用逐元素乘积 ⊙）：
   ∂J/∂z = (∂J/∂h) ⊙ φ'(z)               (结果维度：h × 1)

7. 隐藏层权重 W_1 的完整梯度：
   ∂J/∂W_1 = (∂J/∂z) · xᵀ + λ · W_1      (结果维度：h × d)
```

---

### 四、 训练 vs 预测的内存/显存开销对比

本节指出了深度学习系统设计中的一个核心资源权衡：

1. **为什么训练比预测耗费更多显存？**
   * **反向传播依赖前向缓存**：计算 `∂J/∂W_2` 时必须用到前向阶段的激活值 `h`；计算 `∂J/∂z` 时必须用到未激活输入 `z`。
   * **中间变量必须常驻**：前向计算出来的所有中间变量（activations）必须在显存中保留，直到对应的反向传播步骤执行完毕才能释放。
   * **预测/推理阶段**：无需计算梯度，前向计算完一层后，上一层的激活值即可立即被释放/覆盖，因此显存占用极低。

2. **OOM（Out of Memory）的根本诱因**：
   * 显存中存储的中间变量大小与 **网络深度（层数）** 以及 **批大小（Batch Size）** 成正比。
   * 模型越深、Batch Size 越大，前向缓存的张量体量呈倍数上升，极易引发显存不足。

---

### 五、 核心小结

| 环节 | 计算方向 | 核心任务 | 存储与资源特点 |
| :--- | :--- | :--- | :--- |
| **前向传播** | 输入层 → 输出层 | 计算预测值与目标损失 `J` | 产生并**暂存**所有中间激活值（`z`, `h` 等） |
| **反向传播** | 输出层 → 输入层 | 利用链式法则，自顶向下计算参数梯度 | **读取并消耗**前向暂存值，计算参数梯度后逐步释放 |
| **训练整体** | 交替迭代 | 前向求值 → 反向求导 → 优化器更新参数 | 显存消耗高，受层数与 Batch Size 影响显著 |
| **推理/预测** | 仅单向前向 | 仅计算输出预测值 | 无需反向导数，不保留历史激活值，显存极轻量 |

以下是 [backprop.ipynb](file:///Users/songchao/WorkSpace/PythonProject/DeepLearn/pytorch/chapter_multilayer-perceptrons/backprop.ipynb) 末尾 5 道练习题的详细解答与推导分析：

---

### 练习 1
> **题目**：假设标量函数 `f` 的输入 `X` 是 `n × m` 矩阵。`f` 相对于 `X` 的梯度维数是多少？

**解答**：
* **梯度维数**：同样是 `n × m`（或形状为 `(n, m)`）。
* **原因**：根据微积分中矩阵导数的定义，标量函数 `f` 对矩阵 `X` 的梯度，是由 `f` 对 `X` 中每个独立元素 `X_{i, j}` 的偏导数构成的矩阵：
  ```text
  [∇_X f]_{i, j} = ∂f / ∂X_{i, j}   (其中 i ∈ [1, n], j ∈ [1, m])
  ```
  输入矩阵有多少个元素，梯度矩阵就包含多少个对应的偏导数，因此两者的维度与形状严格保持一致。

---

### 练习 2
> **题目**：向本节中描述的模型的隐藏层添加偏置项（不需要在正则化项中包含偏置项）。
> 1. 画出相应的计算图。
> 2. 推导正向和反向传播方程。

**解答**：

#### 1. 计算图描述
在前向传播中，隐藏层未激活输出 `z` 增加了偏置向量 `b_1 ∈ ℝʰ`：

```text
       ┌───────┐
x  ───►│       │
       │   *   ├──► (W_1·x) ──┐
W_1 ──►│       │              │
       └───────┘          ┌───┴───┐
                          │   +   ├──► z ──► [ φ ] ──► h ──► ...
b_1 ─────────────────────►│       │
                          └───────┘
```
整体计算图逻辑为：
* **输入节点**：`x`, `W_1`, `b_1`, `W_2`, `y`
* **算子与中间节点**：
  * `W_1` 与 `x` 相乘得到 `W_1 · x`；
  * `(W_1 · x)` 与偏置 `b_1` 相加得到 `z`；
  * `z` 经过激活函数 `φ` 得到隐藏激活值 `h`；
  * `h` 与 `W_2` 相乘得到输出 `o`；
  * `o` 与 `y` 计算得到损失 `L`；
  * `W_1` 和 `W_2` 计算正则项 `s`；`L` 与 `s` 相加得到目标 `J`。

#### 2. 前向传播与反向传播方程推导

* **前向传播方程**：
  ```text
  z = W_1 · x + b_1                       (z ∈ ℝʰ, b_1 ∈ ℝʰ)
  h = φ(z)                                (h ∈ ℝʰ)
  o = W_2 · h                             (o ∈ ℝ^q)
  L = l(o, y)
  s = (λ / 2) · (||W_1||_F² + ||W_2||_F²)
  J = L + s
  ```

* **反向传播方程**：
  ```text
  // 顶层及输出层梯度
  ∂J/∂o = ∂L/∂o ∈ ℝ^q
  ∂J/∂W_2 = (∂J/∂o) · hᵀ + λ · W_2

  // 反向传播至隐藏层
  ∂J/∂h = (W_2)ᵀ · (∂J/∂o)
  ∂J/∂z = (∂J/∂h) ⊙ φ'(z)                 (⊙ 表示逐元素乘积)

  // 隐藏层参数梯度
  ∂J/∂W_1 = (∂J/∂z) · xᵀ + λ · W_1
  ∂J/∂b_1 = ∂J/∂z                         (因为 ∂z/∂b_1 为单位阵，且偏置不参与 L2 正则化)
  ```
  > 若输入为一个批量（Batch Size 为 `B`），则偏置项梯度需要沿批次样本维度累加：
  > `∂J/∂b_1 = ∑_{i=1}^B (∂J/∂z⁽ⁱ⁾)`

---

### 练习 3
> **题目**：计算本节所描述的模型，用于训练和预测的内存占用。

**解答**：
设输入维度为 `d`，隐藏神经元数为 `h`，输出维度为 `q`，批次大小为 `B`，每个浮点数占用 `c` 字节（32 位浮点数 `c = 4B`）。

* **模型静态参数量**：
  * 权重 `W_1`（`h × d`）与 `W_2`（`q × h`），基础显存占用为 `(h·d + q·h) · c`。

* **预测/推理阶段（Inference）内存占用**：
  * **常驻内存**：仅需存储模型权重参数。
  * **前向临时激活值**：预测不需要反向传播，各层激活值计算完毕后即可被后续层复用或释放。峰值只需容纳单层/相邻层的缓存，约为：
    ```text
    预测内存 ≈ 权重参数 + 峰值激活值
             ≈ (h·d + q·h)·c + B·max(d, h, q)·c
    ```

* **训练阶段（Training）内存占用**：
  * **模型参数及梯度**：需要保存权重参数，以及与参数等大的梯度张量（共 `2 × 权重大小`）；若使用 SGD 带动量或 Adam，优化器状态还需额外占用 `1~2 × 权重大小`。
  * **中间激活值缓存（Activation Cache）**：反向传播计算 `∂J/∂W_2` 需要用到 `h`，计算 `∂J/∂z` 需要保留 `z`，计算 `∂J/∂W_1` 需要用到 `x`。为了避免重复计算，前向传播必须把各层中间激活值完整保留在显存中直到反向传播完成：
    * 输入 `X`：`B × d`
    * 隐藏层 `Z` 与 `H`：`2 × B × h`
    * 输出 `O`：`B × q`
  * **总结**：
    ```text
    训练内存 ≈ (权重参数 + 梯度 + 优化器状态) + B · (d + 2h + q) · c
    ```
    训练内存显著大于预测内存，且**前向激活值内存与 Batch Size `B` 以及网络层数成正比**。

---

### 练习 4
> **题目**：假设想计算二阶导数。计算图发生了什么？预计计算需要多长时间？

**解答**：

1. **计算图的变化**：
   * 在一阶反向传播中，框架会将导数计算过程本身记录为新的计算图（在 PyTorch 中对应参数 `create_graph=True`）。
   * 当计算二阶导数时，系统需要在**一阶梯度的数据流图之上，再次反向追踪并构建二次求导计算图**。
   * 计算图的节点数和中间边成倍增加，中间算子（如转置、矩阵乘的导数）寿命大幅延长，显存开销将呈爆炸式增长。

2. **预计计算时间**：
   * **完整 Hessian 矩阵（Hessian Matrix）**：
     若模型总参数量为 `P`（本例中 `P = h·d + q·h`），Hessian 矩阵大小为 `P × P`。要得到完整的二阶矩阵，需要对一阶梯度的每个标量分量分别做一次反向传播，总共需要进行 `P` 次反向求导。
     * **计算时间**约为 `P × (一阶反向传播时间)`。对于现代深度网络（参数成千上万甚至数亿），精确计算完整 Hessian 矩阵在时间上是极其缓慢且不可行的。
   * **Hessian-向量积（HVP, Hessian-Vector Product）**：
     若仅需计算 Hessian 矩阵沿某一方向向量 `v` 的乘积 `(∇²J) · v`，可通过标量技巧（对一阶梯度与向量内积 `(∇J)ᵀ·v` 做一次反向传播）。
     * **计算时间**仅需常规反向传播的 **2~3 倍**，在工程优化中较为可行。

---

### 练习 5
> **题目**：假设计算图对当前拥有的 GPU 来说太大了。
> 1. 请试着把它划分到多个 GPU 上。
> 2. 与小批量训练相比，有哪些优点和缺点？

**解答**：

#### 1. 将计算图切分到多个 GPU 上
主要有两种经典切分策略：
* **流水线并行（Pipeline Parallelism / 层间切分）**：
  * 将网络按层垂直划分到不同 GPU。例如：GPU 0 负责输入层到隐藏层（计算 `z = W_1·x + b_1` 及 `h = φ(z)`），然后将激活向量 `h` 跨卡通信发送给 GPU 1；GPU 1 负责后续的输出层和损失计算。
  * 反向传播时，GPU 1 计算出 `∂J/∂h` 发回给 GPU 0，GPU 0 继续完成反向传播。
* **张量并行（Tensor Parallelism / 层内切分）**：
  * 将大的权重矩阵（如 `W_1`）按行或按列拆开放在不同的 GPU 上。各 GPU 分别完成部分矩阵乘法后，通过卡间通信（All-Reduce / All-Gather）聚合结果。

#### 2. 与小批量训练（数据并行 Data Parallelism）对比

* **数据并行（常规小批量多卡切分）**：每张 GPU 各有一份完整模型，仅将 Batch 切小分发给各卡并行计算。

| 对比维度 | 模型/计算图划分（Model/Pipeline Parallelism） | 小批量训练（Data Parallelism） |
| :--- | :--- | :--- |
| **超大模型支持** | **优点**：能够容纳单卡显存无法装下的超大模型，突破物理单卡显存上限。 | **缺点**：若单卡连一个最小模型都放不下，则完全无法运行。 |
| **硬件利用率** | **缺点**：流水线各层存在先后依赖，下层 GPU 经常需要等待上层完成，存在明显的「流水线气泡（Bubble）」，GPU 空闲率高。 | **优点**：各 GPU 计算完全对称，硬件计算单元几乎时刻处于满载状态，吞吐效率高。 |
| **跨卡通信开销** | **缺点**：每层的前向中间激活值与反向梯度必须在卡间频繁传递，对跨卡互联带宽和网络延迟极度敏感。 | **优点**：只需在反向传播结束时同步一次参数梯度（All-Reduce），通信模式成熟且易重叠。 |
| **工程实现复杂度** | **缺点**：需要精细划分计算图、设计调度与通信逻辑，复杂度高。 | **优点**：框架支持极好（如 PyTorch DDP），无需修改模型定义即可即插即用。 |